# Notebook 01 — TextBraTS Dataset Understanding

---

## Objectives

This notebook performs the first pass over the **TextBraTS** dataset — the radiology-report counterpart to the BraTS2020 MRI volumes used by the CV module.

For each of the 369 patients, TextBraTS ships:
- a free-text radiology report (`*_flair_text.txt`)
- a precomputed BERT embedding (`*_flair_text.npy`, shape `(1, 128, 768)` — one 768-d vector per token position, no pooling applied)

This notebook covers:
- Verifying every patient has both files
- Basic report length statistics (words / characters / lines)
- Inspecting the precomputed embeddings purely for sanity (padding behavior, shape consistency)
- A first look at candidate tokenizers (BioBERT / ClinicalBERT / generic BERT)

> ⚠️ **Scope note:** the embeddings inspected here are TextBraTS's *original* precomputed per-token embeddings. They are **not** the final feature vectors used downstream — Notebooks 06a/06b re-extract a single mean-pooled 768-d vector per report directly from the cleaned text using BioBERT/ClinicalBERT. This notebook's embedding checks exist only to understand padding behavior before deciding the pooling strategy.

No train/validation/test split exists yet at this stage — every statistic below is computed across all 369 patients, which is safe since nothing here is a model parameter (no fitting happens in this notebook).


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 1. Setup & Dataset Path


In [2]:
DATASET_DIR = Path("/kaggle/input/datasets/mariammohamed1095/textbrats/TextBraTSData")

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"

print(DATASET_DIR)

/kaggle/input/datasets/mariammohamed1095/textbrats/TextBraTSData


## 2. Patient Inventory

List every patient folder and confirm both the `.txt` report and the `.npy` embedding exist.


In [3]:
patient_dirs = sorted(
    [
        p
        for p in DATASET_DIR.iterdir()
        if p.is_dir()
    ]
)

print(f"Number of patient folders: {len(patient_dirs)}")

Number of patient folders: 369


In [4]:
records = []

for patient_dir in patient_dirs:

    patient_id = patient_dir.name

    txt_file = patient_dir / f"{patient_id}_flair_text.txt"
    npy_file = patient_dir / f"{patient_id}_flair_text.npy"

    records.append(
        {
            "patient_id": patient_id,
            "txt_exists": txt_file.exists(),
            "npy_exists": npy_file.exists(),
            "txt_path": txt_file,
            "npy_path": npy_file,
        }
    )

dataset_df = pd.DataFrame(records)

dataset_df.head()

,patient_id,txt_exists,npy_exists,txt_path,npy_path
0,BraTS20_Training_001,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...
1,BraTS20_Training_002,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...
2,BraTS20_Training_003,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...
3,BraTS20_Training_004,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...
4,BraTS20_Training_005,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...


### 2.1 File Coverage Summary


In [5]:
print("=" * 50)

print("Patients :", len(dataset_df))

print("TXT files :", dataset_df["txt_exists"].sum())

print("NPY files :", dataset_df["npy_exists"].sum())

print()

print("Missing TXT :", (~dataset_df["txt_exists"]).sum())

print("Missing NPY :", (~dataset_df["npy_exists"]).sum())

Patients : 369
TXT files : 369
NPY files : 369

Missing TXT : 0
Missing NPY : 0


## 3. Load Report Text


In [6]:
reports = []

for _, row in dataset_df.iterrows():

    txt = row["txt_path"]

    if txt.exists():

        report = txt.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    else:

        report = ""

    reports.append(report)

dataset_df["report"] = reports

## 4. Report Length Statistics


In [7]:
dataset_df["characters"] = dataset_df["report"].str.len()

dataset_df["words"] = dataset_df["report"].str.split().str.len()

dataset_df["lines"] = dataset_df["report"].str.count("\n") + 1

dataset_df.describe()

,characters,words,lines
count,369.000000,369.000000,369.0
mean,642.441734,96.840108,1.0
std,112.608269,16.931092,0.0
min,348.000000,53.000000,1.0
25%,563.000000,85.000000,1.0
50%,638.000000,96.000000,1.0
75%,714.000000,107.000000,1.0
max,994.000000,156.000000,1.0


### 4.1 Summary Table


In [8]:
summary = pd.DataFrame(
    {
        "Metric": [
            "Patients",
            "Average words",
            "Median words",
            "Minimum words",
            "Maximum words",
            "Average characters",
        ],
        "Value": [
            len(dataset_df),
            dataset_df.words.mean(),
            dataset_df.words.median(),
            dataset_df.words.min(),
            dataset_df.words.max(),
            dataset_df.characters.mean(),
        ],
    }
)

summary

,Metric,Value
0,Patients,369.000000
1,Average words,96.840108
2,Median words,96.000000
3,Minimum words,53.000000
4,Maximum words,156.000000
5,Average characters,642.441734


### 4.2 Longest / Shortest Reports


In [9]:
display(
    dataset_df.nlargest(
        5,
        "words"
    )[["patient_id", "words"]]
)

display(
    dataset_df.nsmallest(
        5,
        "words"
    )[["patient_id", "words"]]
)

,patient_id,words
313,BraTS20_Training_314,156
136,BraTS20_Training_137,153
212,BraTS20_Training_213,146
337,BraTS20_Training_338,140
1,BraTS20_Training_002,139


,patient_id,words
277,BraTS20_Training_278,53
176,BraTS20_Training_177,57
74,BraTS20_Training_075,59
109,BraTS20_Training_110,59
25,BraTS20_Training_026,60


### 4.3 Example Report


In [10]:
example = dataset_df.iloc[0]

print(example.patient_id)

print()

print(example.report)

BraTS20_Training_001

The lesion area is in the right frontal and parietal lobes with a mixed pattern of high and low signals with speckled high signal regions. Edema is mainly observed in the right parietal lobe, partially extending to the frontal lobe, presenting as high signal, indicating significant tissue swelling around the lesion. Necrosis is within the lesions of the right parietal and frontal lobes, appearing as mixed, with alternating high and low signal regions. Ventricular compression is seen in the lateral ventricles with significant compressive effects on the brain tissue and ventricles.


## 5. Embedding Shape Consistency Check

Confirm every patient's precomputed `.npy` embedding has the exact same shape — `(1, 128, 768)` — before relying on it for anything downstream.


In [11]:

embeddings = []
shapes = []

for _, row in dataset_df.iterrows():
    npy_path = row["npy_path"]
    arr = np.load(npy_path, allow_pickle=True)
    embeddings.append(arr)
    shapes.append(arr.shape)

dataset_df["embedding_shape"] = shapes

shape_counts = pd.Series(shapes).value_counts()
print("Embedding shape distribution across all 369 patients:")
print(shape_counts)

assert shape_counts.shape[0] == 1, (
    "Inconsistent embedding shapes found -- investigate before "
    "proceeding to preprocessing design / fusion."
)
print("\nAll embeddings share a single consistent shape - OK")


Embedding shape distribution across all 369 patients:
(1, 128, 768)    369
Name: count, dtype: int64

All embeddings share a single consistent shape - OK


## 6. Padding Behavior — Are [PAD] Positions Zero Vectors?

This is the key check that drives the pooling decision in Notebook 06a/06b: naive mean pooling over all 128 token positions is only safe if padding positions are exactly zero. BERT-family models do **not** guarantee this.


In [12]:

# Confirm [PAD]-position embeddings are NOT zero vectors (normal BERT
# behavior) -- this is exactly why naive mean pooling over all 128
# positions would be unsafe without knowing the real token count.

sample_idx = 0
sample_embedding = embeddings[sample_idx][0]  # (128, 768)
sample_words = dataset_df.loc[sample_idx, "words"]

token_norms = np.linalg.norm(sample_embedding, axis=1)

print(f"Patient: {dataset_df.loc[sample_idx, 'patient_id']}")
print(f"Report word count: {sample_words}")
print(f"\nPer-token L2 norm -- first 10: {token_norms[:10].round(2)}")
print(f"Per-token L2 norm -- last 10:  {token_norms[-10:].round(2)}")
print(f"\nExact-zero rows (would indicate explicit zero-padding): "
      f"{(token_norms < 1e-6).sum()}")


Patient: BraTS20_Training_001
Report word count: 91

Per-token L2 norm -- first 10: [34.78 34.71 35.14 36.23 37.57 33.66 30.33 32.95 27.55 34.17]
Per-token L2 norm -- last 10:  [48.23 46.7  48.67 48.39 48.77 48.2  48.49 48.83 48.7  47.92]

Exact-zero rows (would indicate explicit zero-padding): 0


## 7. Save Inventory

Saved as `textbrats_dataset_inventory.csv` — consumed by Notebook 02 (validation) and Notebook 03 (split application). This file covers all 369 patients; the train/validation/test split is applied later, in Notebook 03, by filtering on `patient_id` against the CV module's `dataset_split.json`.


In [13]:
REPORTS_DIR = Path("/kaggle/working/reports/nlp")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

dataset_df.to_csv(
    REPORTS_DIR / "textbrats_dataset_inventory.csv",
    index=False
)

print("Inventory saved.")

Inventory saved.


## 8. Padding Detection on the Shortest Report

Repeats the padding check from Section 6 on the single shortest report in the dataset — the case most likely to actually reach the [PAD] region.


In [14]:

shortest_idx = dataset_df["words"].idxmin()
shortest_embedding = embeddings[shortest_idx][0]  # (128, 768)
shortest_words = dataset_df.loc[shortest_idx, "words"]
shortest_id = dataset_df.loc[shortest_idx, "patient_id"]

shortest_norms = np.linalg.norm(shortest_embedding, axis=1)

print(f"Shortest report: {shortest_id} ({shortest_words} words)")
print(f"\nAll 128 per-token norms:")
print(shortest_norms.round(2))

# Look for repeated (padding) tail: consecutive near-identical norms
diffs = np.abs(np.diff(shortest_norms))
near_zero_diff = diffs < 0.01
print(f"\nNumber of consecutive near-identical norm pairs (diff < 0.01): "
      f"{near_zero_diff.sum()}")
if near_zero_diff.sum() > 0:
    first_flat = np.argmax(near_zero_diff)
    print(f"First repeated-norm pair starts at token index: {first_flat}")
    print("→ Likely padding starts around this index.")
else:
    print("No repeated tail found → no padding detected, even in the shortest report.")


Shortest report: BraTS20_Training_278 (53 words)

All 128 per-token norms:
[35.5  35.51 38.57 34.71 37.92 32.92 34.52 38.87 37.48 40.22 42.45 35.84
 38.26 41.72 40.64 33.5  31.76 27.98 30.46 32.61 36.88 32.45 33.57 36.1
 42.41 33.23 32.1  28.14 30.39 39.22 41.29 38.51 41.13 36.95 39.69 34.69
 38.38 36.92 30.99 35.32 38.96 37.16 42.05 34.2  33.27 32.77 36.46 30.31
 35.92 33.42 39.94 34.23 30.48 27.82 29.71 34.92 34.87 37.77 41.31 35.89
 32.19 38.76 40.53 31.92 33.24 34.7  32.08 35.12 47.42 48.67 48.24 47.52
 48.42 48.64 48.11 48.39 48.53 48.86 48.87 48.02 48.44 48.57 46.45 47.56
 48.41 43.51 48.4  47.48 48.25 47.03 48.71 48.16 48.08 48.82 48.31 46.73
 46.26 46.58 48.61 48.24 46.11 46.99 46.9  46.66 47.96 47.79 47.81 47.36
 47.53 47.76 47.82 48.63 49.32 47.75 48.2  49.   49.17 46.49 46.73 47.22
 49.36 48.73 48.25 48.78 49.12 47.56 46.78 46.99]

Number of consecutive near-identical norm pairs (diff < 0.01): 1
First repeated-norm pair starts at token index: 0
→ Likely padding starts around

## 9. Candidate Tokenizer Comparison

Quick look at how four candidate tokenizers handle the shortest and longest report — purely exploratory, to sanity-check that none of them would silently truncate badly. The actual max-length decision is made later, in Notebook 05a/05b, using **train-only** statistics (see the fix note there).


In [15]:

longest_idx = dataset_df["words"].idxmax()
longest_text = dataset_df.loc[longest_idx, "report"]
shortest_text = dataset_df.loc[shortest_idx, "report"]

candidate_models = [
    "dmis-lab/biobert-base-cased-v1.1",
    "emilyalsentzer/Bio_ClinicalBERT",
    "bert-base-uncased",
    "bert-base-cased",
]

try:
    from transformers import AutoTokenizer

    for model_name in candidate_models:
        try:
            tok = AutoTokenizer.from_pretrained(model_name)
            n_short = len(tok(shortest_text, truncation=False)["input_ids"])
            n_long  = len(tok(longest_text, truncation=False)["input_ids"])
            print(
                f"{model_name:38s} | shortest: {n_short:4d} tok "
                f"| longest: {n_long:4d} tok "
                f"({'TRUNCATED' if n_long > 128 else 'fits in 128'})"
            )
        except Exception as e:
            print(f"{model_name:38s} -> failed: {e}")

except ImportError:
    print("transformers not installed -- run `pip install transformers`.")


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

dmis-lab/biobert-base-cased-v1.1       | shortest:   79 tok | longest:  203 tok (TRUNCATED)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

bert-base-cased                        | shortest:   79 tok | longest:  203 tok (TRUNCATED)


## 10. Re-verify Embedding Shape Consistency

Same check as Section 5, repeated as a final guard before this notebook's outputs are reused downstream.


In [16]:

embeddings = []
shapes = []

for _, row in dataset_df.iterrows():
    npy_path = row["npy_path"]
    arr = np.load(npy_path, allow_pickle=True)
    embeddings.append(arr)
    shapes.append(arr.shape)

dataset_df["embedding_shape"] = shapes

shape_counts = pd.Series(shapes).value_counts()
print("Embedding shape distribution across all 369 patients:")
print(shape_counts)

assert shape_counts.shape[0] == 1, (
    "Inconsistent embedding shapes found -- investigate before "
    "proceeding to preprocessing design / fusion."
)
print("\nAll embeddings share a single consistent shape - OK")


Embedding shape distribution across all 369 patients:
(1, 128, 768)    369
Name: count, dtype: int64

All embeddings share a single consistent shape - OK
